# Prediksi Minat Beli Pelanggan Online
Nama : Farant Marchelino
NIM  : 2024081012

**Metode yang digunakan:** Naive Bayes Classifier

Pada notebook ini, kita akan mencoba memprediksi apakah seorang pelanggan **tertarik** atau **tidak tertarik** membeli suatu produk secara online. Prediksi dilakukan berdasarkan tiga fitur: harga produk, ketersediaan diskon, dan ulasan produk.

Naive Bayes bekerja dengan menghitung probabilitas setiap kelas berdasarkan data historis, lalu memilih kelas dengan probabilitas tertinggi sebagai hasil prediksi.

---
## Dataset

Berikut adalah data historis dari 8 pelanggan yang akan dijadikan data latih. Setiap baris mewakili satu pelanggan dengan tiga fitur penentu dan label kelasnya.

In [1]:
import pandas as pd

data = {
    'No': [1, 2, 3, 4, 5, 6, 7, 8],
    'Harga': ['Murah', 'Murah', 'Sedang', 'Mahal', 'Mahal', 'Sedang', 'Sedang', 'Mahal'],
    'Diskon': ['Ada', 'Ada', 'Ada', 'Tidak Ada', 'Tidak Ada', 'Ada', 'Tidak Ada', 'Ada'],
    'Ulasan': ['Baik', 'Baik', 'Baik', 'Buruk', 'Buruk', 'Baik', 'Baik', 'Buruk'],
    'Hasil': ['Tertarik', 'Tertarik', 'Tertarik', 'Tidak Tertarik', 'Tidak Tertarik',
              'Tertarik', 'Tertarik', 'Tidak Tertarik']
}

df = pd.DataFrame(data)
df

,No,Harga,Diskon,Ulasan,Hasil
0,1,Murah,Ada,Baik,Tertarik
1,2,Murah,Ada,Baik,Tertarik
2,3,Sedang,Ada,Baik,Tertarik
3,4,Mahal,Tidak Ada,Buruk,Tidak Tertarik
4,5,Mahal,Tidak Ada,Buruk,Tidak Tertarik
5,6,Sedang,Ada,Baik,Tertarik
6,7,Sedang,Tidak Ada,Baik,Tertarik
7,8,Mahal,Ada,Buruk,Tidak Tertarik


**Data pelanggan baru yang ingin diprediksi:**

| Fitur  | Nilai  |
|--------|--------|
| Harga  | Sedang |
| Diskon | Ada    |
| Ulasan | Baik   |

Pertanyaannya: apakah pelanggan ini akan **Tertarik** atau **Tidak Tertarik** membeli produk?

---
##  Langkah 1 — Prior Probability

Prior probability mencerminkan seberapa sering masing-masing kelas muncul dalam data latih. Nilai ini menjadi bobot awal sebelum mempertimbangkan fitur apapun.

$$P(Kelas) = \frac{\text{Jumlah data pada kelas}}{\text{Total data}}$$

In [3]:
total = len(df)
n_tertarik = len(df[df['Hasil'] == 'Tertarik'])
n_tidak    = len(df[df['Hasil'] == 'Tidak Tertarik'])

prior_tertarik = n_tertarik / total
prior_tidak    = n_tidak / total

print(f"Total data: {total}")
print(f"Tertarik: {n_tertarik}, Tidak Tertarik: {n_tidak}")
print()
print(f"P(Tertarik)      = {n_tertarik}/{total} = {prior_tertarik:.4f}")
print(f"P(Tidak Tertarik) = {n_tidak}/{total} = {prior_tidak:.4f}")

Total data: 8
Tertarik: 5, Tidak Tertarik: 3

P(Tertarik)      = 5/8 = 0.6250
P(Tidak Tertarik) = 3/8 = 0.3750


---
## Langkah 2 — Likelihood

Likelihood mengukur seberapa besar kemungkinan suatu nilai fitur muncul pada kelas tertentu. Dihitung secara terpisah untuk setiap fitur dan setiap kelas.

$$P(Fitur = nilai \mid Kelas) = \frac{\text{Jumlah kemunculan nilai fitur pada kelas}}{\text{Jumlah data pada kelas}}$$

Fitur yang dihitung berdasarkan data baru: **Harga = Sedang**, **Diskon = Ada**, **Ulasan = Baik**.

In [4]:
df_t  = df[df['Hasil'] == 'Tertarik']
df_nt = df[df['Hasil'] == 'Tidak Tertarik']

# Kelas Tertarik
p_harga_t  = len(df_t[df_t['Harga']  == 'Sedang']) / n_tertarik
p_diskon_t = len(df_t[df_t['Diskon'] == 'Ada'])    / n_tertarik
p_ulasan_t = len(df_t[df_t['Ulasan'] == 'Baik'])   / n_tertarik

# Kelas Tidak Tertarik
p_harga_nt  = len(df_nt[df_nt['Harga']  == 'Sedang']) / n_tidak
p_diskon_nt = len(df_nt[df_nt['Diskon'] == 'Ada'])    / n_tidak
p_ulasan_nt = len(df_nt[df_nt['Ulasan'] == 'Baik'])   / n_tidak

print("--- Tertarik ---")
print(f"P(Harga=Sedang  | Tertarik) = {len(df_t[df_t['Harga']=='Sedang'])}/{n_tertarik} = {p_harga_t:.4f}")
print(f"P(Diskon=Ada    | Tertarik) = {len(df_t[df_t['Diskon']=='Ada'])}/{n_tertarik} = {p_diskon_t:.4f}")
print(f"P(Ulasan=Baik   | Tertarik) = {len(df_t[df_t['Ulasan']=='Baik'])}/{n_tertarik} = {p_ulasan_t:.4f}")
print()
print("--- Tidak Tertarik ---")
print(f"P(Harga=Sedang  | Tidak Tertarik) = {len(df_nt[df_nt['Harga']=='Sedang'])}/{n_tidak} = {p_harga_nt:.4f}")
print(f"P(Diskon=Ada    | Tidak Tertarik) = {len(df_nt[df_nt['Diskon']=='Ada'])}/{n_tidak} = {p_diskon_nt:.4f}")
print(f"P(Ulasan=Baik   | Tidak Tertarik) = {len(df_nt[df_nt['Ulasan']=='Baik'])}/{n_tidak} = {p_ulasan_nt:.4f}")

--- Tertarik ---
P(Harga=Sedang  | Tertarik) = 3/5 = 0.6000
P(Diskon=Ada    | Tertarik) = 4/5 = 0.8000
P(Ulasan=Baik   | Tertarik) = 5/5 = 1.0000

--- Tidak Tertarik ---
P(Harga=Sedang  | Tidak Tertarik) = 0/3 = 0.0000
P(Diskon=Ada    | Tidak Tertarik) = 1/3 = 0.3333
P(Ulasan=Baik   | Tidak Tertarik) = 0/3 = 0.0000


---
## Langkah 3 — Posterior & Keputusan Akhir

Nilai posterior dihitung dengan mengalikan prior dan semua likelihood untuk masing-masing kelas. Kelas dengan nilai posterior tertinggi dipilih sebagai hasil prediksi.

$$P(Kelas \mid X) \propto P(Kelas) \times P(F_1 \mid Kelas) \times P(F_2 \mid Kelas) \times P(F_3 \mid Kelas)$$

> Tanda $\propto$ berarti *berbanding lurus dengan* — kita tidak menormalisasi nilainya karena hanya perlu membandingkan kedua kelas.

In [6]:
post_tertarik = prior_tertarik * p_harga_t  * p_diskon_t  * p_ulasan_t
post_tidak    = prior_tidak    * p_harga_nt * p_diskon_nt * p_ulasan_nt

print(f"P(Tertarik | X)       = {prior_tertarik:.4f} x {p_harga_t:.4f} x {p_diskon_t:.4f} x {p_ulasan_t:.4f} = {post_tertarik:.4f}")
print(f"P(Tidak Tertarik | X) = {prior_tidak:.4f} x {p_harga_nt:.4f} x {p_diskon_nt:.4f} x {p_ulasan_nt:.4f} = {post_tidak:.4f}")
print()

hasil = 'Tertarik' if post_tertarik > post_tidak else 'Tidak Tertarik'
print(f"Hasil Klasifikasi: {hasil}")

P(Tertarik | X)       = 0.6250 x 0.6000 x 0.8000 x 1.0000 = 0.3000
P(Tidak Tertarik | X) = 0.3750 x 0.0000 x 0.3333 x 0.0000 = 0.0000

Hasil Klasifikasi: Tertarik


---
## Kesimpulan

Berdasarkan perhitungan Naive Bayes, pelanggan dengan profil **Harga = Sedang**, **Diskon = Ada**, dan **Ulasan = Baik** diprediksi sebagai **Tertarik** membeli produk.

**Mengapa hasilnya Tertarik?**

1. **Prior mendukung** — Dari 8 data latih, 5 di antaranya berlabel Tertarik (62,5%), sehingga secara bawaan kelas ini memiliki peluang lebih besar.
2. **Semua likelihood bernilai positif** — Kombinasi Harga Sedang (0.6), Diskon Ada (0.8), dan Ulasan Baik (1.0) semuanya ditemukan dalam data kelas Tertarik.
3. **Kelas Tidak Tertarik gugur** — Karena tidak ada data Tidak Tertarik dengan Harga Sedang maupun Ulasan Baik, nilai likelihood-nya 0, sehingga posteriornya langsung menjadi 0.

> ⚠️ **Catatan:** Kondisi likelihood = 0 disebut *zero probability problem*. Dalam praktiknya, hal ini diatasi menggunakan teknik **Laplace Smoothing** agar tidak ada kelas yang langsung tereliminasi hanya karena satu fitur tidak muncul di data latih.